# Start-to-Finish Cartesian Wave Project

Generate, build, run, and inspect the standalone Cartesian wave project from start to finish.

Navigation: [Index](../index.ipynb) | Previous: [Wave Equation and C Code Generation](wave_equation_and_c_codegen.ipynb) | Next: [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)

## Roadmap

This notebook proceeds through short focused cells, each with one action and one interpretable output.

## Generate the Real Project
The command writes the same project inspected in the Cartesian wave project lesson, now as the complete Cartesian wave workflow.

## Step 1: Import Python helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.

In [1]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile

def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")

def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True, timeout=timeout)
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)

def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError("This notebook requires make to build the generated project.")
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError("This notebook requires a C compiler such as cc, gcc, or clang.")

## Step 2: Create a disposable workspace

The workspace keeps generated files separate from the tutorial source tree.

In [2]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_cartesian_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

## Step 3: Run the generator

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.

In [3]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project path: project/wave_equation_cartesian")

generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
project path: project/wave_equation_cartesian


## Inspect Files and Build

## Step 4: Confirm the Generated Project Directory

Only after generation do we name the generated project files. The inventory is intentionally compact.

In [4]:
required = ["Makefile", "BHaH_function_prototypes.h", "rhs_eval.c", "wave_equation_cartesian.par"]
for relative_path in required:
    path = PROJECT_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)
print("selected generated files:")
for relative_path in required:
    print(relative_path)
for subdir in ["MoL", "diagnostics"]:
    count = sum(1 for path in (PROJECT_DIR / subdir).glob("*.c")) if (PROJECT_DIR / subdir).is_dir() else 0
    print(f"{subdir}/ C files:", count)

selected generated files:
Makefile
BHaH_function_prototypes.h
rhs_eval.c
wave_equation_cartesian.par
MoL/ C files: 3
diagnostics/ C files: 0


## Step 5: Inspect the Parameter File

The parameter file is the runtime interface for the generated executable.

In [5]:
print((PROJECT_DIR / "wave_equation_cartesian.par").read_text(encoding="utf-8", errors="replace"))

#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1       # (int)



## Step 6: Inspect Build Rules

Only the leading Makefile lines are needed here; the anatomy notebook returns to file structure later.

In [6]:
for line in (PROJECT_DIR / "Makefile").read_text(encoding="utf-8", errors="replace").splitlines()[:35]:
    print(line)

CC ?= gcc  # assigns the value CC to gcc only if environment variable CC is not already set

CFLAGS = -std=gnu99 -O2 -march=native -g -Wall -I.
CXXFLAGS = -I. -O2 -g -Wall -Wno-unknown-pragmas -march=native
VALGRIND_CFLAGS = -I. -std=gnu99 -O2 -g -Wall -Wno-unknown-pragmas
INCLUDEDIRS =
LDFLAGS = -lm
# Check for OpenMP support
OPENMP_FLAG = -fopenmp
COMPILER_SUPPORTS_OPENMP := $(shell echo | $(CC) $(OPENMP_FLAG) -E - >/dev/null 2>&1 && echo YES || echo NO)

ifeq ($(COMPILER_SUPPORTS_OPENMP), YES)
    CFLAGS += $(OPENMP_FLAG)
    LDFLAGS += $(OPENMP_FLAG)
endif

OBJ_FILES = apply_bcs.o cmdline_input_and_parfile_parser.o commondata_struct_set_to_default.o diagnostics.o exact_solution_single_Cartesian_point.o griddata_free.o initial_data.o main.o MoL/MoL_free_intermediate_stage_gfs.o MoL/MoL_malloc_intermediate_stage_gfs.o MoL/MoL_step_forward_in_time.o numerical_grids_and_timestep.o params_struct_set_to_default.o progress_indicator.o rhs_eval.o

all: wave_equation_cartesian

%.o: %.c $(C

## Step 7: Inspect Function Prototypes

The prototype header lists callable generated functions.

In [7]:
for line in (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(encoding="utf-8", errors="replace").splitlines():
    if "rhs_eval" in line or "diagnostics" in line or line.startswith("void "):
        print(line)

void apply_bcs(const commondata_struct *restrict commondata, const params_struct *restrict params, REAL *restrict gfs);
void cmdline_input_and_parfile_parser(commondata_struct *restrict commondata, int argc, const char *argv[]);
void commondata_struct_set_to_default(commondata_struct *restrict commondata);
void diagnostics(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
void exact_solution_single_Cartesian_point(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL xCart0,
void griddata_free(commondata_struct *restrict commondata, griddata_struct *restrict griddata,
void initial_data(const commondata_struct *restrict commondata, griddata_struct *restrict griddata);
void MoL_free_intermediate_stage_gfs(MoL_gridfunctions_struct *restrict gridfuncs);
void MoL_malloc_intermediate_stage_gfs(const commondata_struct *restrict commondata, const params_struct *restrict params,
void MoL_step_forward_in_time(commondata_struct *

## Step 8: Inspect the RHS Kernel Excerpt

This excerpt shows the generated update kernel without turning the first project lesson into file anatomy.

In [8]:
lines = (PROJECT_DIR / "rhs_eval.c").read_text(encoding="utf-8", errors="replace").splitlines()
for line in lines[:60]:
    print(line)

#include "BHaH_defines.h"
#include "intrinsics/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void rhs_eval(const commondata_struct *restrict commondata, const params_struct *restrict params, const REAL *restrict in_gfs,
              REAL *restrict rhs_gfs) {
#include "set_CodeParameters-simd.h"
#pragma omp parallel for
  for (int i2 = NGHOSTS; i2 < Nxx_plus_2NGHOSTS2 - NGHOSTS; i2++) {
    for (int i1 = NGHOSTS; i1 < Nxx_plus_2NGHOSTS1 - NGHOSTS; i1++) {
      for (int i0 = NGHOSTS; i0 < Nxx_plus_2NGHOSTS0 - NGHOSTS; i0 += SIMD_WIDTH) {

        /*
         * NRPy-Generated GF Access/FD Code, Step 1 of 2:
         * Read gridfunction(s) from main memory and compute FD stencils as needed.
         */
        const REAL_SIMD_ARRAY uu_i2m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 2)]);
        const REAL_SIMD_ARRAY uu_i2m1 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1, i2 - 1)]);
        const REAL_SIMD_ARRAY uu_i1m2 = ReadSIMD(&in_gfs[IDX4(UUGF, i0, i1 - 2, i2)]);
        const RE

## Run and Interpret Diagnostics
The run is shortened for notebook execution, but it still exercises the generated executable and diagnostic writer.

## Step 9: Shorten runtime parameters

Only runtime values are changed so the notebook run finishes quickly.

In [9]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace("diagnostics_output_every = 0.2", "diagnostics_output_every = 0.1")
par_text = par_text.replace("output_progress_every = 1", "output_progress_every = 1000000")
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

--- runtime wave_equation_cartesian.par ---
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.1  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 0.2                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1000000       # (int)



## Step 10: Build the executable

The build step compiles generated C after checking that external build tools are available.

In [10]:
require_toolchain()
build_output = run_command(["make", "-j2"], PROJECT_DIR, timeout=300)
print("build completed")
print("compiler output line count:", len(build_output.splitlines()))

build completed
compiler output line count: 16


## Step 11: Run the Default Resolution

The default run writes the first diagnostic file.

In [11]:
default_output = run_command([f"./{PROJECT_NAME}"], PROJECT_DIR, timeout=90)
print("run output (default resolution):")
for line in default_output.splitlines()[:12]:
    if line.strip():
        print(line.rstrip())

run output (default resolution):
It: 0 t=0.000 / 0.2 = 0.00% dt=1/6.4 | t/h=0.00 ETA 0h00m00s


## Step 12: Run a Refined Resolution

The convergence-factor run repeats the same executable at higher resolution.

In [12]:
refined_output = run_command([f"./{PROJECT_NAME}", "2.0"], PROJECT_DIR, timeout=90)
print("run output (convergence factor 2.0):")
for line in refined_output.splitlines()[:12]:
    if line.strip():
        print(line.rstrip())

run output (convergence factor 2.0):
It: 0 t=0.000 / 0.2 = 0.00% dt=1/12.8 | t/h=0.00 ETA 0h00m00s


## Step 13: Run the focused calculation

The output is used directly in the next step.

In [13]:
diagnostic_rows = {}
for diagnostic in sorted(PROJECT_DIR.glob("out0d-conv_factor*.txt")):
    rows = [[float(value) for value in line.split()] for line in diagnostic.read_text(encoding="utf-8", errors="replace").splitlines() if line.strip()]
    if len(rows) < 2:
        raise RuntimeError(f"Expected at least two data rows in {diagnostic.name}.")
    diagnostic_rows[diagnostic.name] = rows
    print(diagnostic.name, "rows:", len(rows), "last row:", rows[-1])
coarse = diagnostic_rows["out0d-conv_factor1.00.txt"][-1][1]
fine = diagnostic_rows["out0d-conv_factor2.00.txt"][-1][1]
print("last-row uu error ratio coarse/fine:", coarse / fine)
if fine >= coarse:
    raise RuntimeError("Expected the refined run to reduce the uu error.")

out0d-conv_factor1.00.txt rows: 2 last row: [0.15625, 4.061498e-08, 2.140944e-05, 3.983805, 3.983805]
out0d-conv_factor2.00.txt rows: 2 last row: [0.078125, 6.435354e-10, 1.354916e-06, 3.995936, 3.995936]
last-row uu error ratio coarse/fine: 63.11227012531089


## Interpreting the Output
The runtime parameter file matches the displayed diagnostics. The two diagnostic files show the same Cartesian wave project running at two resolutions after generation and compilation.

## Where This Leads
- [Boundary Conditions and Convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb)
- [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)